In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.retail_ops;
CREATE VOLUME IF NOT EXISTS workspace.retail_ops.raw;

In [0]:
%python
display(dbutils.fs.ls("/Volumes/workspace/retail_ops/raw"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/retail_ops/raw/orders.csv,orders.csv,97115,1777878541000
dbfs:/Volumes/workspace/retail_ops/raw/returns.csv,returns.csv,11904,1777878541000
dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv,shipments.csv,71531,1777878541000


In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema_prefix", "retail_ops")
dbutils.widgets.text("raw_path", "/Volumes/workspace/retail_ops/raw")
dbutils.widgets.text("run_id", "")

catalog = dbutils.widgets.get("catalog")
schema_prefix = dbutils.widgets.get("schema_prefix")
raw_path = dbutils.widgets.get("raw_path")
run_id = dbutils.widgets.get("run_id")

landing_schema = f"{catalog}.{schema_prefix}"
bronze_schema = f"{catalog}.{schema_prefix}_bronze"
silver_schema = f"{catalog}.{schema_prefix}_silver"
gold_schema = f"{catalog}.{schema_prefix}_gold"
ops_schema = f"{catalog}.{schema_prefix}_ops"

for schema_name in [landing_schema, bronze_schema, silver_schema, gold_schema, ops_schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print(f"catalog       : {catalog}")
print(f"landing schema: {landing_schema}")
print(f"raw path      : {raw_path}")
print(f"bronze schema : {bronze_schema}")
print(f"silver schema : {silver_schema}")
print(f"gold schema   : {gold_schema}")
print(f"ops schema    : {ops_schema}")
print(f"run id        : {run_id if run_id else 'autogenerated in downstream notebooks'}")

catalog       : workspace
landing schema: workspace.retail_ops
raw path      : /Volumes/workspace/retail_ops/raw
bronze schema : workspace.retail_ops_bronze
silver schema : workspace.retail_ops_silver
gold schema   : workspace.retail_ops_gold
ops schema    : workspace.retail_ops_ops
run id        : autogenerated in downstream notebooks


In [0]:
from pyspark.sql import functions as F


def ingest_csv(file_name: str, table_name: str) -> None:
    source_path = f"{raw_path}/{file_name}"

    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(source_path)
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("source_file_name", F.col("_metadata.file_path"))
    )

    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{bronze_schema}.{table_name}")
    )

    print(f"loaded {file_name} -> {bronze_schema}.{table_name}")


ingest_csv("orders.csv", "orders_bronze")
ingest_csv("shipments.csv", "shipments_bronze")
ingest_csv("returns.csv", "returns_bronze")

# COMMAND ----------

for table_name in ["orders_bronze", "shipments_bronze", "returns_bronze"]:
    display(spark.table(f"{bronze_schema}.{table_name}").limit(5))

loaded orders.csv -> workspace.retail_ops_bronze.orders_bronze
loaded shipments.csv -> workspace.retail_ops_bronze.shipments_bronze
loaded returns.csv -> workspace.retail_ops_bronze.returns_bronze


order_id,order_date,customer_id,customer_name,segment,city,state,region,product_category,product_name,quantity,unit_price,discount_pct,promised_delivery_days,payment_mode,record_updated_ts,bronze_ingested_at,source_file_name
ORD00001,2025-01-27,CUST0076,Delta Stores,Consumer,Bengaluru,Karnataka,SOUTH,GROCERY,Healthy Snack Box,3,899.00,5%,2,CASH ON DELIVERY,2025-01-27 21:47:00,2026-05-04T07:59:56.034Z,dbfs:/Volumes/workspace/retail_ops/raw/orders.csv
ORD00002,03/13/2025,CUST0014,Orbit Outlet,null,Kolkata,West Bengal,east,Fashion,winter jacket,1,3512.00,15,3,UPI,2025-03-13 04:13:00,2026-05-04T07:59:56.034Z,dbfs:/Volumes/workspace/retail_ops/raw/orders.csv
ORD00003,30-06-2025,CUST0099,city cart,Consumer,JAIPUR,Rajasthan,North,fashion,Winter Jacket,5,3719.00,20%,2,net banking,2025-06-30 02:02:00,2026-05-04T07:59:56.034Z,dbfs:/Volumes/workspace/retail_ops/raw/orders.csv
ORD00004,2025/03/04,CUST0035,Green Grocer,Corporate,bhubaneswar,Odisha,EAST,Fashion,winter jacket,1,INR 3972.00,5 %,5,UPI,2025-03-04 05:29:00,2026-05-04T07:59:56.034Z,dbfs:/Volumes/workspace/retail_ops/raw/orders.csv
ORD00005,06/14/2025,CUST0018,SMART CHOICE,Home Office,Guwahati,null,East,Fashion,backpack,2,INR 1920.00,15%,4,Cash on Delivery,2025-06-14 14:09:00,2026-05-04T07:59:56.034Z,dbfs:/Volumes/workspace/retail_ops/raw/orders.csv


shipment_id,order_id,warehouse_id,carrier,shipping_mode,ship_date,delivery_date,shipping_cost,delivery_status,record_updated_ts,bronze_ingested_at,source_file_name
SHP00001,ORD00001,CCU-01,ecom express,Same Day,27-01-2025,02-02-2025,233.57,DELIVERED,2025-01-27 04:00:00,2026-05-04T07:59:58.775Z,dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv
SHP00002,ORD00002,CCU-01,XpressBees,standard,2025-03-15,18-03-2025,INR 327.48,delivered,2025-03-15 13:00:00,2026-05-04T07:59:58.775Z,dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv
SHP00003,ORD00003,MUM-01,XPRESSBEES,Standard,2025-07-02,05-07-2025,286.72,Delivered,2025-07-02 05:00:00,2026-05-04T07:59:58.775Z,dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv
SHP00004,ORD00004,DEL-01,xpressbees,standard,2025-03-04,10-03-2025,INR 176.06,delivered,2025-03-04 05:00:00,2026-05-04T07:59:58.775Z,dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv
SHP00005,ord00005,MUM-01,Blue Dart,same day,2025-06-15,18-06-2025,INR 113.95,DELIVERED,2025-06-15 15:00:00,2026-05-04T07:59:58.775Z,dbfs:/Volumes/workspace/retail_ops/raw/shipments.csv


return_id,order_id,return_date,return_reason,refund_amount,returned_units,resolution_status,record_updated_ts,bronze_ingested_at,source_file_name
RTN00001,ORD00561,04/26/2025,delay issue,INR 3059.41,1,Refunded,2025-04-26 01:00:00,2026-05-04T08:00:01.733Z,dbfs:/Volumes/workspace/retail_ops/raw/returns.csv
RTN00002,ORD00008,2025-05-14,wrong product,2522.34,1,Refunded,2025-05-14 17:00:00,2026-05-04T08:00:01.733Z,dbfs:/Volumes/workspace/retail_ops/raw/returns.csv
RTN00003,ORD00384,2025/05/14,Product Damaged,914.03,2,Investigation Closed,2025-05-14 05:00:00,2026-05-04T08:00:01.733Z,dbfs:/Volumes/workspace/retail_ops/raw/returns.csv
RTN00004,ord00480,12-06-2025,delivered late,INR 1426.94,2,REPLACEMENT SENT,2025-06-12 12:00:00,2026-05-04T08:00:01.733Z,dbfs:/Volumes/workspace/retail_ops/raw/returns.csv
RTN00005,ORD00051,2025-05-22,Wrong Item,INR 3599.06,2,Refunded,2025-05-22 12:00:00,2026-05-04T08:00:01.733Z,dbfs:/Volumes/workspace/retail_ops/raw/returns.csv


In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F


CITY_TO_STATE = {
    "Bengaluru": "Karnataka",
    "Hyderabad": "Telangana",
    "Chennai": "Tamil Nadu",
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Ahmedabad": "Gujarat",
    "Delhi": "Delhi",
    "Noida": "Uttar Pradesh",
    "Jaipur": "Rajasthan",
    "Kolkata": "West Bengal",
    "Bhubaneswar": "Odisha",
    "Guwahati": "Assam",
}

STATE_TO_REGION = {
    "Karnataka": "South",
    "Telangana": "South",
    "Tamil Nadu": "South",
    "Maharashtra": "West",
    "Gujarat": "West",
    "Delhi": "North",
    "Uttar Pradesh": "North",
    "Rajasthan": "North",
    "West Bengal": "East",
    "Odisha": "East",
    "Assam": "East",
}


def parse_date(column_name: str) -> F.Column:
    column = F.trim(F.col(column_name))
    return F.coalesce(
        F.try_to_date(column, F.lit("yyyy-MM-dd")),
        F.try_to_date(column, F.lit("dd-MM-yyyy")),
        F.try_to_date(column, F.lit("MM/dd/yyyy")),
        F.try_to_date(column, F.lit("yyyy/MM/dd")),
    )


def parse_timestamp(column_name: str) -> F.Column:
    return F.try_to_timestamp(F.col(column_name), F.lit("yyyy-MM-dd HH:mm:ss"))


def parse_currency(column_name: str) -> F.Column:
    cleaned = F.regexp_replace(F.trim(F.col(column_name)), r"[^0-9.\-]", "")
    return F.when(cleaned == "", None).otherwise(cleaned.cast("double"))


def parse_percent(column_name: str) -> F.Column:
    cleaned = F.regexp_replace(F.lower(F.trim(F.col(column_name))), r"[^0-9.\-]", "")
    numeric_value = F.when(cleaned == "", None).otherwise(cleaned.cast("double"))
    return F.when(numeric_value > 1, numeric_value / 100).otherwise(numeric_value)


def title_case(column_name: str) -> F.Column:
    return F.initcap(F.lower(F.trim(F.col(column_name))))


orders_bronze = spark.table(f"{bronze_schema}.orders_bronze")
shipments_bronze = spark.table(f"{bronze_schema}.shipments_bronze")
returns_bronze = spark.table(f"{bronze_schema}.returns_bronze")

# COMMAND ----------

orders_window = Window.partitionBy(F.upper(F.trim(F.col("order_id")))).orderBy(
    parse_timestamp("record_updated_ts").desc_nulls_last()
)

orders_silver = (
    orders_bronze.withColumn("order_id_clean", F.upper(F.trim(F.col("order_id"))))
    .withColumn("order_date_clean", parse_date("order_date"))
    .withColumn("record_updated_ts_clean", parse_timestamp("record_updated_ts"))
    .withColumn("customer_name_clean", title_case("customer_name"))
    .withColumn("segment_clean", F.when(F.trim(F.col("segment")) == "", "Consumer").otherwise(title_case("segment")))
    .withColumn("city_clean", title_case("city"))
    .withColumn("state_clean", title_case("state"))
    .withColumn(
        "state_clean",
        F.when(
            F.col("state_clean").isNull() | (F.col("state_clean") == ""),
            F.create_map([F.lit(item) for pair in CITY_TO_STATE.items() for item in pair]).getItem(F.col("city_clean")),
        ).otherwise(F.col("state_clean")),
    )
    .withColumn("region_clean", title_case("region"))
    .withColumn(
        "region_clean",
        F.when(
            F.col("region_clean").isNull() | (F.col("region_clean") == ""),
            F.create_map([F.lit(item) for pair in STATE_TO_REGION.items() for item in pair]).getItem(F.col("state_clean")),
        ).otherwise(F.col("region_clean")),
    )
    .withColumn("product_category_clean", title_case("product_category"))
    .withColumn("product_name_clean", title_case("product_name"))
    .withColumn("quantity_clean", F.col("quantity").cast("int"))
    .withColumn("unit_price_clean", parse_currency("unit_price"))
    .withColumn("discount_pct_clean", parse_percent("discount_pct"))
    .withColumn(
        "discount_pct_clean",
        F.when(
            (F.col("discount_pct_clean").isNull()) | (F.col("discount_pct_clean") < 0) | (F.col("discount_pct_clean") > 0.60),
            F.lit(0.0),
        ).otherwise(F.col("discount_pct_clean")),
    )
    .withColumn("promised_delivery_days_clean", F.col("promised_delivery_days").cast("int"))
    .withColumn("payment_mode_clean", title_case("payment_mode"))
    .withColumn("row_num", F.row_number().over(orders_window))
    .filter(F.col("row_num") == 1)
    .select(
        F.col("order_id_clean").alias("order_id"),
        F.col("order_date_clean").alias("order_date"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.col("customer_name_clean").alias("customer_name"),
        F.col("segment_clean").alias("segment"),
        F.col("city_clean").alias("city"),
        F.col("state_clean").alias("state"),
        F.col("region_clean").alias("region"),
        F.col("product_category_clean").alias("product_category"),
        F.col("product_name_clean").alias("product_name"),
        F.col("quantity_clean").alias("quantity"),
        F.round(F.col("unit_price_clean"), 2).alias("unit_price"),
        F.round(F.col("discount_pct_clean"), 4).alias("discount_pct"),
        F.col("promised_delivery_days_clean").alias("promised_delivery_days"),
        F.col("payment_mode_clean").alias("payment_mode"),
        F.col("record_updated_ts_clean").alias("record_updated_ts"),
        F.round(F.col("quantity_clean") * F.col("unit_price_clean"), 2).alias("gross_amount"),
        F.round(F.col("quantity_clean") * F.col("unit_price_clean") * F.col("discount_pct_clean"), 2).alias("discount_value"),
        F.round(
            (F.col("quantity_clean") * F.col("unit_price_clean"))
            - (F.col("quantity_clean") * F.col("unit_price_clean") * F.col("discount_pct_clean")),
            2,
        ).alias("net_order_amount"),
    )
)

# COMMAND ----------

shipments_window = Window.partitionBy(F.upper(F.trim(F.col("order_id")))).orderBy(
    parse_timestamp("record_updated_ts").desc_nulls_last()
)

shipments_base = (
    shipments_bronze.withColumn("order_id_clean", F.upper(F.trim(F.col("order_id"))))
    .withColumn("shipment_id_clean", F.upper(F.trim(F.col("shipment_id"))))
    .withColumn("ship_date_clean", parse_date("ship_date"))
    .withColumn("delivery_date_clean", parse_date("delivery_date"))
    .withColumn("shipping_cost_clean", parse_currency("shipping_cost"))
    .withColumn(
        "shipping_cost_clean",
        F.when(F.col("shipping_cost_clean") < 0, None).otherwise(F.col("shipping_cost_clean")),
    )
    .withColumn("carrier_clean", title_case("carrier"))
    .withColumn("shipping_mode_clean", title_case("shipping_mode"))
    .withColumn("delivery_status_clean", title_case("delivery_status"))
    .withColumn("record_updated_ts_clean", parse_timestamp("record_updated_ts"))
)

carrier_average_cost = shipments_base.groupBy("carrier_clean").agg(
    F.round(F.avg("shipping_cost_clean"), 2).alias("carrier_avg_shipping_cost")
)

shipments_silver = (
    shipments_base.join(carrier_average_cost, on="carrier_clean", how="left")
    .withColumn(
        "shipping_cost_final",
        F.coalesce(F.col("shipping_cost_clean"), F.col("carrier_avg_shipping_cost"), F.lit(0.0)),
    )
    .withColumn("row_num", F.row_number().over(shipments_window))
    .filter(F.col("row_num") == 1)
    .select(
        F.col("shipment_id_clean").alias("shipment_id"),
        F.col("order_id_clean").alias("order_id"),
        F.trim(F.col("warehouse_id")).alias("warehouse_id"),
        F.col("carrier_clean").alias("carrier"),
        F.col("shipping_mode_clean").alias("shipping_mode"),
        F.col("ship_date_clean").alias("ship_date"),
        F.col("delivery_date_clean").alias("delivery_date"),
        F.round(F.col("shipping_cost_final"), 2).alias("shipping_cost"),
        F.col("delivery_status_clean").alias("delivery_status"),
        F.col("record_updated_ts_clean").alias("record_updated_ts"),
    )
)

# COMMAND ----------

returns_window = Window.partitionBy(F.upper(F.trim(F.col("order_id")))).orderBy(
    parse_timestamp("record_updated_ts").desc_nulls_last()
)

returns_silver = (
    returns_bronze.withColumn("return_id_clean", F.upper(F.trim(F.col("return_id"))))
    .withColumn("order_id_clean", F.upper(F.trim(F.col("order_id"))))
    .withColumn("return_date_clean", parse_date("return_date"))
    .withColumn("refund_amount_clean", parse_currency("refund_amount"))
    .withColumn("returned_units_clean", F.col("returned_units").cast("int"))
    .withColumn("resolution_status_clean", title_case("resolution_status"))
    .withColumn("record_updated_ts_clean", parse_timestamp("record_updated_ts"))
    .withColumn(
        "return_reason_clean",
        F.when(F.lower(F.col("return_reason")).contains("damage"), "Damaged")
        .when(F.lower(F.col("return_reason")).contains("wrong"), "Wrong Item")
        .when(F.lower(F.col("return_reason")).contains("size"), "Size Issue")
        .when(F.lower(F.col("return_reason")).contains("fit"), "Size Issue")
        .when(F.lower(F.col("return_reason")).contains("late"), "Late Delivery")
        .when(F.lower(F.col("return_reason")).contains("delay"), "Late Delivery")
        .when(F.lower(F.col("return_reason")).contains("quality"), "Quality Issue")
        .when(F.lower(F.col("return_reason")).contains("mind"), "No Longer Needed")
        .when(F.lower(F.col("return_reason")).contains("need"), "No Longer Needed")
        .otherwise("Other"),
    )
    .withColumn("row_num", F.row_number().over(returns_window))
    .filter(F.col("row_num") == 1)
    .select(
        F.col("return_id_clean").alias("return_id"),
        F.col("order_id_clean").alias("order_id"),
        F.col("return_date_clean").alias("return_date"),
        F.col("return_reason_clean").alias("return_reason"),
        F.round(F.col("refund_amount_clean"), 2).alias("refund_amount"),
        F.when(F.col("returned_units_clean") < 1, 1).otherwise(F.col("returned_units_clean")).alias("returned_units"),
        F.col("resolution_status_clean").alias("resolution_status"),
        F.col("record_updated_ts_clean").alias("record_updated_ts"),
    )
)

# COMMAND ----------

(orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.orders_silver"))
(shipments_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.shipments_silver"))
(returns_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.returns_silver"))

print(f"saved {silver_schema}.orders_silver")
print(f"saved {silver_schema}.shipments_silver")
print(f"saved {silver_schema}.returns_silver")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/column.py:527: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


saved workspace.retail_ops_silver.orders_silver
saved workspace.retail_ops_silver.shipments_silver
saved workspace.retail_ops_silver.returns_silver


In [0]:



# COMMAND ----------

from pyspark.sql import functions as F


orders_silver = spark.table(f"{silver_schema}.orders_silver")
shipments_silver = spark.table(f"{silver_schema}.shipments_silver")
returns_silver = spark.table(f"{silver_schema}.returns_silver")

order_service_mart = (
    orders_silver.alias("o")
    .join(shipments_silver.alias("s"), on="order_id", how="left")
    .join(returns_silver.alias("r"), on="order_id", how="left")
    .withColumn(
        "actual_delivery_days",
        F.when(
            F.col("s.delivery_date").isNotNull() & F.col("s.ship_date").isNotNull(),
            F.datediff(F.col("s.delivery_date"), F.col("s.ship_date")),
        ),
    )
    .withColumn(
        "delay_days",
        F.when(
            F.col("actual_delivery_days").isNotNull(),
            F.greatest(F.col("actual_delivery_days") - F.col("o.promised_delivery_days"), F.lit(0)),
        ).otherwise(F.lit(None)),
    )
    .withColumn(
        "on_time_flag",
        F.when(
            (F.col("s.delivery_status") == "Delivered")
            & (F.col("actual_delivery_days") <= F.col("o.promised_delivery_days")),
            F.lit(1),
        )
        .when(F.col("s.delivery_status") == "Delivered", F.lit(0))
        .otherwise(F.lit(None)),
    )
    .withColumn("returned_flag", F.when(F.col("r.return_id").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("refund_amount", F.coalesce(F.col("r.refund_amount"), F.lit(0.0)))
    .withColumn("shipping_cost", F.coalesce(F.col("s.shipping_cost"), F.lit(0.0)))
    .withColumn(
        "net_realized_revenue",
        F.round(F.col("o.net_order_amount") - F.col("refund_amount") - F.col("shipping_cost"), 2),
    )
    .withColumn(
        "order_lifecycle_status",
        F.when(F.col("returned_flag") == 1, "Returned")
        .when(F.col("s.delivery_status") == "Delivered", "Delivered")
        .when(F.col("s.delivery_status").isNull(), "No Shipment Record")
        .otherwise(F.col("s.delivery_status")),
    )
    .select(
        "order_id",
        F.col("o.order_date").alias("order_date"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.customer_name").alias("customer_name"),
        F.col("o.segment").alias("segment"),
        F.col("o.city").alias("city"),
        F.col("o.state").alias("state"),
        F.col("o.region").alias("region"),
        F.col("o.product_category").alias("product_category"),
        F.col("o.product_name").alias("product_name"),
        F.col("o.quantity").alias("quantity"),
        F.col("o.unit_price").alias("unit_price"),
        F.col("o.discount_pct").alias("discount_pct"),
        F.col("o.gross_amount").alias("gross_amount"),
        F.col("o.discount_value").alias("discount_value"),
        F.col("o.net_order_amount").alias("net_order_amount"),
        F.col("o.promised_delivery_days").alias("promised_delivery_days"),
        F.col("o.payment_mode").alias("payment_mode"),
        F.col("s.shipment_id").alias("shipment_id"),
        F.col("s.warehouse_id").alias("warehouse_id"),
        F.col("s.carrier").alias("carrier"),
        F.col("s.shipping_mode").alias("shipping_mode"),
        F.col("s.ship_date").alias("ship_date"),
        F.col("s.delivery_date").alias("delivery_date"),
        F.col("s.delivery_status").alias("delivery_status"),
        "shipping_cost",
        "actual_delivery_days",
        "delay_days",
        "on_time_flag",
        F.col("r.return_id").alias("return_id"),
        F.col("r.return_date").alias("return_date"),
        F.col("r.return_reason").alias("return_reason"),
        "refund_amount",
        F.col("r.returned_units").alias("returned_units"),
        F.col("r.resolution_status").alias("resolution_status"),
        "returned_flag",
        "order_lifecycle_status",
        "net_realized_revenue",
    )
)

(
    order_service_mart.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.order_service_mart")
)

print(f"saved {gold_schema}.order_service_mart")
display(order_service_mart.limit(10))

saved workspace.retail_ops_gold.order_service_mart


order_id,order_date,customer_id,customer_name,segment,city,state,region,product_category,product_name,quantity,unit_price,discount_pct,gross_amount,discount_value,net_order_amount,promised_delivery_days,payment_mode,shipment_id,warehouse_id,carrier,shipping_mode,ship_date,delivery_date,delivery_status,shipping_cost,actual_delivery_days,delay_days,on_time_flag,return_id,return_date,return_reason,refund_amount,returned_units,resolution_status,returned_flag,order_lifecycle_status,net_realized_revenue
ORD00001,2025-01-27,CUST0076,Delta Stores,Consumer,Bengaluru,Karnataka,South,Grocery,Healthy Snack Box,3,899.0,0.05,2697.0,134.85,2562.15,2,Cash On Delivery,SHP00001,CCU-01,Ecom Express,Same Day,2025-01-27,2025-02-02,Delivered,233.57,6,4,0,RTN00065,2025-01-30,Wrong Item,1679.35,1,Investigation Closed,1,Returned,649.23
ORD00002,2025-03-13,CUST0014,Orbit Outlet,null,Kolkata,West Bengal,East,Fashion,Winter Jacket,1,3512.0,0.15,3512.0,526.8,2985.2,3,Upi,SHP00002,CCU-01,Xpressbees,Standard,2025-03-15,2025-03-18,Delivered,327.48,3,0,1,null,null,null,0.0,null,null,0,Delivered,2657.72
ORD00003,2025-06-30,CUST0099,City Cart,Consumer,Jaipur,Rajasthan,North,Fashion,Winter Jacket,5,3719.0,0.2,18595.0,3719.0,14876.0,2,Net Banking,SHP00003,MUM-01,Xpressbees,Standard,2025-07-02,2025-07-05,Delivered,286.72,3,1,0,null,null,null,0.0,null,null,0,Delivered,14589.28
ORD00004,2025-03-04,CUST0035,Green Grocer,Corporate,Bhubaneswar,Odisha,East,Fashion,Winter Jacket,1,3972.0,0.05,3972.0,198.6,3773.4,5,Upi,SHP00004,DEL-01,Xpressbees,Standard,2025-03-04,2025-03-10,Delivered,176.06,6,1,0,RTN00100,2025-03-15,Other,2254.12,2,Investigation Closed,1,Returned,1343.22
ORD00005,2025-06-14,CUST0018,Smart Choice,Home Office,Guwahati,Assam,East,Fashion,Backpack,2,1920.0,0.15,3840.0,576.0,3264.0,4,Cash On Delivery,SHP00005,MUM-01,Blue Dart,Same Day,2025-06-15,2025-06-18,Delivered,113.95,3,0,1,null,null,null,0.0,null,null,0,Delivered,3150.05
ORD00006,2025-01-17,CUST0068,Bright Mart,Consumer,Hyderabad,Telangana,South,Electronics,Smart Watch,2,5445.0,0.15,10890.0,1633.5,9256.5,5,Cash On Delivery,SHP00006,MUM-01,Blue Dart,Express,2025-01-18,2025-01-22,Delivered,68.58,4,0,1,null,null,null,0.0,null,null,0,Delivered,9187.92
ORD00007,2025-06-13,CUST0020,Indus Shop,Corporate,Jaipur,Rajasthan,North,Electronics,Smart Watch,5,4857.0,0.1,24285.0,2428.5,21856.5,7,Net Banking,SHP00007,CCU-01,Ecom Express,Same Day,2025-06-15,2025-06-26,Delivered,120.25,11,4,0,RTN00014,2025-06-25,Late Delivery,656.09,1,Investigation Closed,1,Returned,21080.16
ORD00008,2025-05-05,CUST0017,Happy Homes,Corporate,Bengaluru,Karnataka,South,Fashion,Running Shoes,5,2629.0,0.0,13145.0,0.0,13145.0,6,Net Banking,SHP00008,BLR-01,Blue Dart,Same Day,2025-05-06,2025-05-16,Delivered,185.58,10,4,0,RTN00002,2025-05-14,Wrong Item,2522.34,1,Refunded,1,Returned,10437.08
ORD00009,2025-01-06,CUST0076,Quick Basket,Consumer,Noida,Uttar Pradesh,North,Electronics,Wireless Earbuds,2,3179.0,0.0,6358.0,0.0,6358.0,3,Card,SHP00009,CCU-01,Xpressbees,Standard,2025-01-07,2025-01-14,Delivered,232.82,7,4,0,null,null,null,0.0,null,null,0,Delivered,6125.18
ORD00010,2025-05-02,CUST0104,Prime Square,Corporate,Mumbai,Maharashtra,West,Home,Storage Rack,2,2889.0,0.2,5778.0,1155.6,4622.4,5,Upi,SHP00010,MUM-01,Shadowfax,Same Day,2025-05-02,2025-05-07,Delivered,318.12,5,0,1,RTN00081,2025-05-09,Size Issue,1758.87,1,Investigation Closed,1,Returned,2545.41


In [0]:


# COMMAND ----------

import uuid
from pyspark.sql import functions as F


effective_run_id = run_id if run_id else str(uuid.uuid4())
run_timestamp = F.current_timestamp()

orders_bronze = spark.table(f"{bronze_schema}.orders_bronze")
shipments_bronze = spark.table(f"{bronze_schema}.shipments_bronze")
returns_bronze = spark.table(f"{bronze_schema}.returns_bronze")

orders_silver = spark.table(f"{silver_schema}.orders_silver")
shipments_silver = spark.table(f"{silver_schema}.shipments_silver")
returns_silver = spark.table(f"{silver_schema}.returns_silver")

order_service_mart = spark.table(f"{gold_schema}.order_service_mart")


def parse_date(column_name: str) -> F.Column:
    column = F.trim(F.col(column_name))
    return F.coalesce(
        F.try_to_date(column, F.lit("yyyy-MM-dd")),
        F.try_to_date(column, F.lit("dd-MM-yyyy")),
        F.try_to_date(column, F.lit("MM/dd/yyyy")),
        F.try_to_date(column, F.lit("yyyy/MM/dd")),
    )


def parse_currency(column_name: str) -> F.Column:
    cleaned = F.regexp_replace(F.trim(F.col(column_name)), r"[^0-9.\-]", "")
    return F.when(cleaned == "", None).otherwise(cleaned.cast("double"))


orders_invalid = (
    orders_bronze.withColumn("parsed_order_date", parse_date("order_date"))
    .withColumn("parsed_unit_price", parse_currency("unit_price"))
    .withColumn("parsed_quantity", F.col("quantity").cast("int"))
    .withColumn(
        "dq_issue",
        F.when(F.trim(F.col("order_id")) == "", "missing_order_id")
        .when(F.col("parsed_order_date").isNull(), "invalid_order_date")
        .when(F.col("parsed_quantity").isNull() | (F.col("parsed_quantity") <= 0), "invalid_quantity")
        .when(F.col("parsed_unit_price").isNull() | (F.col("parsed_unit_price") <= 0), "invalid_unit_price")
        .otherwise(None),
    )
    .filter(F.col("dq_issue").isNotNull())
    .withColumn("run_id", F.lit(effective_run_id))
    .withColumn("captured_at", run_timestamp)
)

shipments_invalid = (
    shipments_bronze.withColumn("parsed_ship_date", parse_date("ship_date"))
    .withColumn("parsed_delivery_date", parse_date("delivery_date"))
    .withColumn("parsed_shipping_cost", parse_currency("shipping_cost"))
    .withColumn(
        "dq_issue",
        F.when(F.trim(F.col("shipment_id")) == "", "missing_shipment_id")
        .when(F.trim(F.col("order_id")) == "", "missing_order_id")
        .when(F.col("parsed_ship_date").isNull(), "invalid_ship_date")
        .when(
            F.col("parsed_delivery_date").isNotNull() & (F.col("parsed_delivery_date") < F.col("parsed_ship_date")),
            "delivery_before_ship_date",
        )
        .when(F.col("parsed_shipping_cost").isNull(), "invalid_shipping_cost")
        .otherwise(None),
    )
    .filter(F.col("dq_issue").isNotNull())
    .withColumn("run_id", F.lit(effective_run_id))
    .withColumn("captured_at", run_timestamp)
)

returns_invalid = (
    returns_bronze.withColumn("parsed_return_date", parse_date("return_date"))
    .withColumn("parsed_refund_amount", parse_currency("refund_amount"))
    .withColumn("parsed_returned_units", F.col("returned_units").cast("int"))
    .withColumn(
        "dq_issue",
        F.when(F.trim(F.col("return_id")) == "", "missing_return_id")
        .when(F.trim(F.col("order_id")) == "", "missing_order_id")
        .when(F.col("parsed_return_date").isNull(), "invalid_return_date")
        .when(F.col("parsed_refund_amount").isNull() | (F.col("parsed_refund_amount") < 0), "invalid_refund_amount")
        .when(F.col("parsed_returned_units").isNull() | (F.col("parsed_returned_units") <= 0), "invalid_returned_units")
        .otherwise(None),
    )
    .filter(F.col("dq_issue").isNotNull())
    .withColumn("run_id", F.lit(effective_run_id))
    .withColumn("captured_at", run_timestamp)
)

(
    orders_invalid.write.format("delta")
    .mode("append")
    .saveAsTable(f"{ops_schema}.orders_invalid")
)

(
    shipments_invalid.write.format("delta")
    .mode("append")
    .saveAsTable(f"{ops_schema}.shipments_invalid")
)

(
    returns_invalid.write.format("delta")
    .mode("append")
    .saveAsTable(f"{ops_schema}.returns_invalid")
)

print(f"saved {ops_schema}.orders_invalid")
print(f"saved {ops_schema}.shipments_invalid")
print(f"saved {ops_schema}.returns_invalid")
print(f"run_id: {effective_run_id}")

saved workspace.retail_ops_ops.orders_invalid
saved workspace.retail_ops_ops.shipments_invalid
saved workspace.retail_ops_ops.returns_invalid
run_id: 30df816f-bdba-4dd8-b881-97127197590e


In [0]:

# COMMAND ----------

from pyspark.sql import Window
from pyspark.sql import functions as F


order_service_mart = spark.table(f"{gold_schema}.order_service_mart")

dim_date = (
    order_service_mart.select(F.col("order_date").alias("date_value"))
    .where(F.col("date_value").isNotNull())
    .distinct()
    .withColumn("date_key", F.date_format("date_value", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date_value"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("date_value")))
    .withColumn("month_number", F.month("date_value"))
    .withColumn("month_name", F.date_format("date_value", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date_value"))
    .withColumn("day_of_month", F.dayofmonth("date_value"))
)

customer_window = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc_nulls_last(), F.col("order_id").desc())

dim_customer = (
    order_service_mart.select("customer_id", "customer_name", "segment", "city", "state", "region", "order_date", "order_id")
    .where(F.col("customer_id").isNotNull())
    .withColumn("row_num", F.row_number().over(customer_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "order_date", "order_id")
    .withColumn("customer_key", F.abs(F.xxhash64("customer_id")).cast("long"))
)

dim_product = (
    order_service_mart.select("product_category", "product_name")
    .where(F.col("product_name").isNotNull())
    .distinct()
    .withColumn("product_key", F.abs(F.xxhash64("product_category", "product_name")).cast("long"))
)

dim_carrier = (
    order_service_mart.select("carrier", "shipping_mode", "warehouse_id")
    .where(F.col("carrier").isNotNull())
    .distinct()
    .withColumn("carrier_key", F.abs(F.xxhash64("carrier", "shipping_mode", "warehouse_id")).cast("long"))
)

fact_order_fulfillment = (
    order_service_mart.alias("f")
    .join(dim_date.alias("d"), F.col("f.order_date") == F.col("d.date_value"), "left")
    .join(dim_customer.alias("c"), F.col("f.customer_id") == F.col("c.customer_id"), "left")
    .join(
        dim_product.alias("p"),
        (F.col("f.product_category") == F.col("p.product_category")) & (F.col("f.product_name") == F.col("p.product_name")),
        "left",
    )
    .join(
        dim_carrier.alias("r"),
        (F.col("f.carrier") == F.col("r.carrier"))
        & (F.col("f.shipping_mode") == F.col("r.shipping_mode"))
        & (F.col("f.warehouse_id") == F.col("r.warehouse_id")),
        "left",
    )
    .select(
        F.col("f.order_id").alias("order_id"),
        F.col("d.date_key").alias("order_date_key"),
        F.col("c.customer_key").alias("customer_key"),
        F.col("p.product_key").alias("product_key"),
        F.col("r.carrier_key").alias("carrier_key"),
        F.col("f.shipment_id").alias("shipment_id"),
        F.col("f.return_id").alias("return_id"),
        F.col("f.quantity").alias("quantity"),
        F.col("f.unit_price").alias("unit_price"),
        F.col("f.discount_pct").alias("discount_pct"),
        F.col("f.gross_amount").alias("gross_amount"),
        F.col("f.discount_value").alias("discount_value"),
        F.col("f.net_order_amount").alias("net_order_amount"),
        F.col("f.shipping_cost").alias("shipping_cost"),
        F.col("f.refund_amount").alias("refund_amount"),
        F.col("f.net_realized_revenue").alias("net_realized_revenue"),
        F.col("f.promised_delivery_days").alias("promised_delivery_days"),
        F.col("f.actual_delivery_days").alias("actual_delivery_days"),
        F.col("f.delay_days").alias("delay_days"),
        F.col("f.on_time_flag").alias("on_time_flag"),
        F.col("f.returned_flag").alias("returned_flag"),
        F.col("f.order_lifecycle_status").alias("order_lifecycle_status"),
        F.col("f.return_reason").alias("return_reason"),
        F.col("f.delivery_status").alias("delivery_status"),
    )
)

# COMMAND ----------

(dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_date"))
(dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_customer"))
(dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_product"))
(dim_carrier.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_carrier"))
(fact_order_fulfillment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_order_fulfillment"))

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {gold_schema}.vw_powerbi_dim_date AS
    SELECT * FROM {gold_schema}.dim_date
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {gold_schema}.vw_powerbi_dim_customer AS
    SELECT * FROM {gold_schema}.dim_customer
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {gold_schema}.vw_powerbi_dim_product AS
    SELECT * FROM {gold_schema}.dim_product
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {gold_schema}.vw_powerbi_dim_carrier AS
    SELECT * FROM {gold_schema}.dim_carrier
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE VIEW {gold_schema}.vw_powerbi_fact_order_fulfillment AS
    SELECT * FROM {gold_schema}.fact_order_fulfillment
    """
)

print(f"saved {gold_schema}.dim_date")
print(f"saved {gold_schema}.dim_customer")
print(f"saved {gold_schema}.dim_product")
print(f"saved {gold_schema}.dim_carrier")
print(f"saved {gold_schema}.fact_order_fulfillment")

# COMMAND ----------

display(spark.table(f"{gold_schema}.dim_date").limit(10))
display(spark.table(f"{gold_schema}.dim_customer").limit(10))
display(spark.table(f"{gold_schema}.fact_order_fulfillment").limit(10))
display(
    spark.sql(
        f"""
        SELECT
            COUNT(*) AS customer_rows,
            COUNT(DISTINCT customer_id) AS distinct_customer_ids,
            COUNT(DISTINCT customer_key) AS distinct_customer_keys
        FROM {gold_schema}.dim_customer
        """
    )
)
(dim_carrier.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_carrier")
)

(fact_order_fulfillment.write
    .format("delta")
    .mode("overwrite")
   workspace.retail_ops_gold.dim_dateworkspace.retail_ops_gold.dim_date .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_order_fulfillment")
)


saved workspace.retail_ops_gold.dim_date
saved workspace.retail_ops_gold.dim_customer
saved workspace.retail_ops_gold.dim_product
saved workspace.retail_ops_gold.dim_carrier
saved workspace.retail_ops_gold.fact_order_fulfillment


date_value,date_key,year,quarter,month_number,month_name,week_of_year,day_of_month
2025-02-18,20250218,2025,Q1,2,February,8,18
2025-02-20,20250220,2025,Q1,2,February,8,20
2025-03-19,20250319,2025,Q1,3,March,12,19
2025-01-10,20250110,2025,Q1,1,January,2,10
2025-04-11,20250411,2025,Q2,4,April,15,11
2025-03-24,20250324,2025,Q1,3,March,13,24
2025-02-03,20250203,2025,Q1,2,February,6,3
2025-01-11,20250111,2025,Q1,1,January,2,11
2025-03-02,20250302,2025,Q1,3,March,9,2
2025-02-16,20250216,2025,Q1,2,February,7,16


customer_id,customer_name,segment,city,state,region,customer_key
CUST0001,Orbit Outlet,Corporate,Delhi,Delhi,North,2961339139205797322
CUST0002,Happy Homes,Consumer,Hyderabad,Telangana,South,5953474034703390640
CUST0003,Jupiter Market,Home Office,Hyderabad,Telangana,South,6894171027052760544
CUST0004,Orbit Outlet,null,Noida,Uttar Pradesh,North,525195038782421876
CUST0005,Lotus Living,null,Kolkata,West Bengal,East,1381463463241493221
CUST0006,Metro Needs,null,Jaipur,Rajasthan,North,7297145389714770570
CUST0007,Royal House,Home Office,Hyderabad,Telangana,South,5671657191088428483
CUST0008,Elite Bazaar,Corporate,Guwahati,Assam,East,1909493564847624162
CUST0009,Aarav Retail,null,Ahmedabad,Gujarat,West,1730154795696668756
CUST0010,Green Grocer,Corporate,Jaipur,Rajasthan,North,5193154407225212353


order_id,order_date_key,customer_key,product_key,carrier_key,shipment_id,return_id,quantity,unit_price,discount_pct,gross_amount,discount_value,net_order_amount,shipping_cost,refund_amount,net_realized_revenue,promised_delivery_days,actual_delivery_days,delay_days,on_time_flag,returned_flag,order_lifecycle_status,return_reason,delivery_status
ORD00001,20250127,4676166570082249897,8442457865817176117,975827567257409622,SHP00001,RTN00065,3,899.0,0.05,2697.0,134.85,2562.15,233.57,1679.35,649.23,2,6,4,0,1,Returned,Wrong Item,Delivered
ORD00002,20250313,7765139716093944368,2572511397764824448,561893138120344408,SHP00002,null,1,3512.0,0.15,3512.0,526.8,2985.2,327.48,0.0,2657.72,3,3,0,1,0,Delivered,null,Delivered
ORD00003,20250630,4177158156043788400,2572511397764824448,4987539743890998559,SHP00003,null,5,3719.0,0.2,18595.0,3719.0,14876.0,286.72,0.0,14589.28,2,3,1,0,0,Delivered,null,Delivered
ORD00004,20250304,6507231700093938582,2572511397764824448,2555401397545651075,SHP00004,RTN00100,1,3972.0,0.05,3972.0,198.6,3773.4,176.06,2254.12,1343.22,5,6,1,0,1,Returned,Other,Delivered
ORD00005,20250614,1025512047300478772,9189186577057594085,8686370684329944607,SHP00005,null,2,1920.0,0.15,3840.0,576.0,3264.0,113.95,0.0,3150.05,4,3,0,1,0,Delivered,null,Delivered
ORD00006,20250117,7953025032606098720,6555042850806193714,4991175526076245990,SHP00006,null,2,5445.0,0.15,10890.0,1633.5,9256.5,68.58,0.0,9187.92,5,4,0,1,0,Delivered,null,Delivered
ORD00007,20250613,3843743776427446241,6555042850806193714,975827567257409622,SHP00007,RTN00014,5,4857.0,0.1,24285.0,2428.5,21856.5,120.25,656.09,21080.16,7,11,4,0,1,Returned,Late Delivery,Delivered
ORD00008,20250505,6184329307569112667,3849966876416626556,7288334268420664678,SHP00008,RTN00002,5,2629.0,0.0,13145.0,0.0,13145.0,185.58,2522.34,10437.08,6,10,4,0,1,Returned,Wrong Item,Delivered
ORD00009,20250106,4676166570082249897,7070655001327940431,561893138120344408,SHP00009,null,2,3179.0,0.0,6358.0,0.0,6358.0,232.82,0.0,6125.18,3,7,4,0,0,Delivered,null,Delivered
ORD00010,20250502,4015853431967766462,1662142295093862321,189804734544312428,SHP00010,RTN00081,2,2889.0,0.2,5778.0,1155.6,4622.4,318.12,1758.87,2545.41,5,5,0,1,1,Returned,Size Issue,Delivered


customer_rows,distinct_customer_ids,distinct_customer_keys
120,120,120
